In [24]:
import pandas as pd
import json

In [2]:
from sentence_transformers import CrossEncoder

# Load a well-known reranker
model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Your query
query = "What causes dry eyes?"

# Top-k documents retrieved from ChromaDB (example strings)
retrieved_docs = [
    "Dry eyes may result from screen exposure.",
    "Vitamin D helps with bone health.",
    "Eye strain is reduced by blinking frequently."
]

# Prepare input pairs (query, doc)
pairs = [(query, doc) for doc in retrieved_docs]

# Get relevance scores from cross-encoder
scores = model.predict(pairs)

# Rank by score
reranked = sorted(zip(retrieved_docs, scores), key=lambda x: x[1], reverse=True)

# Print reranked results
for doc, score in reranked:
    print(f"{score:.3f} - {doc}")


/Users/rodericktabalba/miniforge3/envs/house_finance_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


7.756 - Dry eyes may result from screen exposure.
-7.318 - Eye strain is reduced by blinking frequently.
-11.168 - Vitamin D helps with bone health.


In [26]:
evaluation = pd.read_json("../policy_eval_500_200_evaluation_k3.json")
results = evaluation["results"]

documents = pd.read_json("../documents/policies.json")

In [29]:
import json

output_file = "results.jsonl"

for result in results:
    query = result["question"]
    print("Query: ", query)
    retrieved_document_ids = set(result["retrieved_document_ids"])
    sentences = []

    for answer in result["retrieved_chunks"]:
        retrieved_document = documents[documents['url'] == answer["metadata"]["document_url"]]
        print("Retrieved document: ", len(retrieved_document))
        print(retrieved_document['extracted'])

        text = retrieved_document['extracted'].values[0] if not retrieved_document.empty else ""
        sentences.extend(str(text).split("."))

    print("Sentences: ", len(sentences))
    pairs = [(query, doc.strip()) for doc in sentences if doc.strip()]

    if not pairs:
        continue

    scores = model.predict(pairs)
    reranked = sorted(zip(sentences, scores), key=lambda x: x[1], reverse=True)

    for doc, score in reranked[:3]:
        prediction = {
            "query": query,
            "document": doc.strip(),
            "score": float(score)
        }
        print(f"{score:.3f} - {doc.strip()}")

        # Append directly to file
        with open(output_file, "a") as f:
            f.write(json.dumps(prediction) + "\n")


Query:  What are the minimum credit hour requirements for an academic major and an academic minor, respectively?
Retrieved document:  1
267    Board of Regents Policy, RP 5.201 \nInstructio...
Name: extracted, dtype: object
Retrieved document:  1
0    #  Executive Policy 5.205 Executive Policy 5.2...
Name: extracted, dtype: object
Retrieved document:  1
0    #  Executive Policy 5.205 Executive Policy 5.2...
Name: extracted, dtype: object
Sentences:  309
3.588 - Minors generally contain fifteen 
to eighteen hours of coursework with at least nine hours of upper-division coursework 
within a specific academic major
0.249 - Academic Minor: Recognition of work completed in select credit courses as a student’s secondary declared academic field of study
0.249 - Academic Minor: Recognition of work completed in select credit courses as a student’s secondary declared academic field of study
Query:  According to section II. Definitions, what distinguishes a "concentration" from an "academic major

KeyboardInterrupt: 